# 07 — Binary and multiclass classification

**Level:** Intermediate · **Study time:** 75–90 minutes (including experiments)

**Prerequisites:** Notebooks 05–06.

## What you will be able to do
- Train with logits and the appropriate classification loss.
- Convert logits into probabilities and predictions.
- Inspect a decision boundary and a confusion matrix.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 06](06_datasets_and_dataloaders.ipynb) · [Next: 08](08_generalization.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Solve a binary problem

A **logit** is an unconstrained score. `BCEWithLogitsLoss` combines sigmoid with binary cross entropy stably. It expects floating targets with the same shape as logits. Do not apply sigmoid before this loss.


In [ ]:
X = torch.randn(500, 2)
y = (X[:, 0] * X[:, 1] > 0).float().unsqueeze(1)
train, val = torch.arange(400), torch.arange(400, 500)
binary = nn.Sequential(nn.Linear(2, 24), nn.ReLU(), nn.Linear(24, 1))
opt = torch.optim.Adam(binary.parameters(), lr=0.03)
loss_fn = nn.BCEWithLogitsLoss()
for _ in range(250):
    opt.zero_grad(set_to_none=True)
    loss = loss_fn(binary(X[train]), y[train])
    loss.backward(); opt.step()
binary.eval()
with torch.no_grad():
    probabilities = binary(X[val]).sigmoid()
    predicted = (probabilities >= 0.5).float()
    binary_accuracy = (predicted == y[val]).float().mean().item()
print("Binary validation accuracy:", binary_accuracy)
assert binary_accuracy > 0.85


## Step 3 — Visualize the boundary

The network combines straight boundaries into a nonlinear decision region. Points near the boundary can be uncertain; a probability threshold controls the final yes/no decision.


In [ ]:
axis = torch.linspace(-3, 3, 100)
gx, gy = torch.meshgrid(axis, axis, indexing="ij")
grid = torch.stack([gx.flatten(), gy.flatten()], dim=1)
with torch.no_grad():
    field = binary(grid).sigmoid().reshape(100, 100)
plt.contourf(gx.numpy(), gy.numpy(), field.numpy(), levels=20, alpha=0.65)
plt.scatter(X[val, 0], X[val, 1], c=y[val, 0], edgecolors="white", s=25)
plt.xlabel("Feature 1"); plt.ylabel("Feature 2"); plt.show()


## Step 4 — Move to mutually exclusive classes

For C classes, `CrossEntropyLoss` expects logits `[B,C]` and integer class indices `[B]` with dtype `long`. It includes log-softmax internally. Softmax is useful for displaying probabilities, but not before this loss.


In [ ]:
centers = torch.tensor([[-2., -1.], [2., -1.], [0., 2.]])
labels = torch.arange(3).repeat_interleave(180)
features = centers[labels] + 0.7 * torch.randn(540, 2)
order = torch.randperm(540)
tr, va = order[:420], order[420:]
multi = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 3))
optimizer = torch.optim.Adam(multi.parameters(), lr=0.03)
for _ in range(120):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(multi(features[tr]), labels[tr])
    loss.backward(); optimizer.step()
multi.eval()
with torch.no_grad():
    logits = multi(features[va])
    predictions = logits.argmax(1)
    accuracy = (predictions == labels[va]).float().mean().item()
print("Multiclass validation accuracy:", accuracy)
confusion = torch.bincount(labels[va] * 3 + predictions, minlength=9).reshape(3, 3)
print("Rows = true class; columns = prediction:", confusion)
assert accuracy > 0.9
assert confusion.sum() == len(va)


## Your turn — practice before opening the solutions

1. Verify that each multiclass probability vector sums to 1.
2. Compute binary precision and recall at thresholds 0.3 and 0.7. Hint: guard against zero denominators.
3. Compute each class's recall from the confusion matrix.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
assert torch.allclose(logits.softmax(1).sum(1), torch.ones(len(va)), atol=1e-6)
truth = y[val].bool()
for threshold in [0.3, 0.7]:
    decision = probabilities >= threshold
    tp = (decision & truth).sum().item()
    precision = tp / max(decision.sum().item(), 1)
    recall = tp / max(truth.sum().item(), 1)
    print(threshold, "precision", precision, "recall", recall)
print("Per-class recall:", confusion.diag() / confusion.sum(1).clamp_min(1))


## Before moving on

Match the loss to the target meaning and shape. Multilabel classification differs from multiclass classification: several labels can be true at once, so independent binary logits are often appropriate.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- [BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)


[Course guide](../README.md) · [Previous: 06](06_datasets_and_dataloaders.ipynb) · [Next: 08](08_generalization.ipynb)
